**An Encoder-Decoder for Neural Machine Translation**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset
import tokenizers

In [2]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name = "eng-spa",
    split=['validation','test']
)
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)

nmt_train_set, nmt_valid_set = split['train'], split['test']

each sample in the dataset is a dictionary containing english text along with its spanish translation.

In [3]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

we will need to tokenize this text. we could use a different tokenizer for englosh and spanish, but, these 2 languages have many words in common (e.g. animal, color, hotel, idea, radio, motor), and many similar subwords (e.g. pre, auto, inter, uni), so it makes sense to use a common tokenizer. Let's train a BPE tokenizer on all the training text, both english and spanish

In [10]:
def train_eng_spa(): # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair['source_text']
        yield pair['target_text']

In [11]:
max_length = 256
vocab_size = 10_000

# building ur own tokenizer
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["<pad>","<unk>","<s>","</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

Let's test this tokenizer:

In [12]:
nmt_tokenizer.encode("I like soccer").ids

[43, 401, 4381]

In [13]:
nmt_tokenizer.encode("<s> Me gusta el fùtbol").ids

[2, 396, 582, 219, 69, 1, 3075]

Perfect, now lets create a small utility class to hold the tokenized english texts (i.e. the source token id sequences), along with the corresponding tokenized spanish targets (target token id sequences), plus the corresponding attention masks. we create a namedtuple base class for this (ie. a tuple with named fields), and extend it to add a to() method, which will make it easy to move tensors to GPU

In [14]:
from collections import namedtuple

In [15]:
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device), self.tgt_token_ids.to(device), self.tgt_mask.to(device))

next, create the dataloaders

In [16]:
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])

    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

In [17]:
batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size, collate_fn=nmt_collate_fn)

In [21]:
# building our model now
# the decoder and encoder will share an nn.Embedding Layer (as they share tokenizer)

class NmtModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id=0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths = src_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden_states = self.encoder(src_packed)
        outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0, 2, 1)

In [23]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [40]:
torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device)

In [41]:
xentropy = nn.CrossEntropyLoss(ignore_index=0) # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
epochs = 30

train_loss = [0] * epochs
val_loss = [0] * epochs

for epoch in range(epochs):
    nmt_model.train()
    for x_batch, y_batch in nmt_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # zero gradients
        optimizer.zero_grad()
        # forward pass
        out = nmt_model(x_batch)
        # loss
        loss = xentropy(out, y_batch)
        # backpropagation
        loss.backward()
        # optimize
        optimizer.step()

        train_loss[epoch] += loss.item() * y_batch.shape[0]
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # evaluation
    nmt_model.eval()
    with torch.no_grad():
        for x_batch, y_batch in nmt_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            out = nmt_model(x_batch)
            loss = xentropy(out, y_batch)
            val_loss[epoch] += loss.item() * y_batch.shape[0]
        val_loss[epoch] /= len(nmt_valid_loader.dataset)
    
    print(f"Epoch: {epoch+1} | Train Loss: {train_loss[epoch]:.4f} | Val Loss: {val_loss[epoch]:.4f}")


Epoch: 1 | Train Loss: 3.1346 | Val Loss: 2.4186
Epoch: 2 | Train Loss: 2.0410 | Val Loss: 2.2304
Epoch: 3 | Train Loss: 1.7223 | Val Loss: 2.2113
Epoch: 4 | Train Loss: 1.5614 | Val Loss: 2.2484
Epoch: 5 | Train Loss: 1.4719 | Val Loss: 2.2918
Epoch: 6 | Train Loss: 1.4168 | Val Loss: 2.3430
Epoch: 7 | Train Loss: 1.3933 | Val Loss: 2.3918
Epoch: 8 | Train Loss: 1.3780 | Val Loss: 2.4339
Epoch: 9 | Train Loss: 1.3773 | Val Loss: 2.4772


KeyboardInterrupt: 

helper function to use the model to translate english text to spanish.
- we start by using the model with the english text for the encoder and a single sos text for the decoder. the decoder will output logits for the first token in the translation. the function will the pick the most likely token (i.e. the token with the highest logit) and add it to the decoder inputs and call the model again to get the next oken. repeating the process, adding one token at a time. 

In [42]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text, "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1) # find the best token ids
            next_token_id = Y_token_ids[0, index] # take the last token id
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [43]:
nmt_model.eval()
translate(nmt_model, "I like soccer")

' Me gusta el fútbol . </s>'

In [44]:
longer_text = "I like to play soccer with my friends."
translate(nmt_model, longer_text)

' Me gusta jugar al golf con los amigos . </s>'

**Improvements:**
- Consider using bidirectional recurrent networks for the encoder
- Consider using beam search, which isn't a change to the model architecture but a way of making the output such that to probabilistically obtain the best output

**Attention Mechanisms:**

In [45]:
# implementing luong's attention
def attention(query, key, value): # note: dq == dk (for the dot product to work) and Lk == Lv
    scores = query @ key.transpose(1,2) # [B, Lq, dq] @ [B, dk, Lk] = [B, Lq, Lk]
    weights = torch.softmax(scores, dim=-1) # [B, Lq, Lk]
    return weights @ value # [B, Lq, Lk] @ [B, Lv, dv] = [B, Lq, dv]

Key:
- B = batch size
- Lq = max query length in this batch
- Lk = max key length in this batch = Lv = max value length in this batch
- dq = query embedding size = dk = key embedding size
- dv = value embedding size

In [50]:
class NmtModelWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, pad_id = 0, hidden_dim=512, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.decoder = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.output = nn.Linear(2*hidden_dim, vocab_size)
    
    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)
        src_lengths = pair.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(
            src_embeddings, lengths=src_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        encoder_outputs_packed, hidden_states = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_states)
        encoder_outputs, _ = pad_packed_sequence(encoder_outputs_packed, batch_first=True)
        attn_output = attention(query=decoder_outputs, key=encoder_outputs, value=encoder_outputs)
        combined_output = torch.cat((attn_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0, 2, 1)

In [51]:
torch.manual_seed(42)
nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)

In [52]:
xentropy = nn.CrossEntropyLoss(ignore_index=0) # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_attn_model.parameters(), lr=0.001)
epochs = 10

train_loss = [0] * epochs
val_loss = [0] * epochs

for epoch in range(epochs):
    nmt_attn_model.train()
    for x_batch, y_batch in nmt_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # zero gradients
        optimizer.zero_grad()
        # forward pass
        out = nmt_attn_model(x_batch)
        # loss
        loss = xentropy(out, y_batch)
        # backpropagation
        loss.backward()
        # optimize
        optimizer.step()

        train_loss[epoch] += loss.item() * y_batch.shape[0]
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # evaluation
    nmt_attn_model.eval()
    with torch.no_grad():
        for x_batch, y_batch in nmt_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            out = nmt_attn_model(x_batch)
            loss = xentropy(out, y_batch)
            val_loss[epoch] += loss.item() * y_batch.shape[0]
        val_loss[epoch] /= len(nmt_valid_loader.dataset)
    
    print(f"Epoch: {epoch+1} | Train Loss: {train_loss[epoch]:.4f} | Val Loss: {val_loss[epoch]:.4f}")


Epoch: 1 | Train Loss: 3.0094 | Val Loss: 2.4209
Epoch: 2 | Train Loss: 2.1274 | Val Loss: 2.3034
Epoch: 3 | Train Loss: 1.9265 | Val Loss: 2.2974
Epoch: 4 | Train Loss: 1.8347 | Val Loss: 2.3438
Epoch: 5 | Train Loss: 1.7862 | Val Loss: 2.3845
Epoch: 6 | Train Loss: 1.7586 | Val Loss: 2.4089
Epoch: 7 | Train Loss: 1.7431 | Val Loss: 2.4493
Epoch: 8 | Train Loss: 1.7353 | Val Loss: 2.4851
Epoch: 9 | Train Loss: 1.7256 | Val Loss: 2.5134
Epoch: 10 | Train Loss: 1.7199 | Val Loss: 2.5478
